In [0]:
create or replace temp view 78HCO_engaged_profiled as
with base_table as (
  select *
  from (
    select * from values
      ('1467525790'),
      ('1023105400'),
      ('1649347469'),
      ('1144266024'),
      ('1003878539'),
      ('1760476659'),
      ('1578693321'),
      ('1396882205'),
      ('1639370059'),
      ('1326092404'),
      ('1295789907'),
      ('1013143213'),
      ('1912939703'),
      ('1346297843'),
      ('1235234535'),
      ('1053632463'),
      ('1376544320'),
      ('1144548322'),
      ('1750482022'),
      ('1598784555'),
      ('1659877280'),
      ('1265694442'),
      ('1013924182'),
      ('1093808040'),
      ('1285174649'),
      ('1114969169'),
      ('1932280666'),
      ('1063702785'),
      ('1164426896'),
      ('1669683512'),
      ('1093894131'),
      ('1215921457'),
      ('1548212988'),
      ('1437365186'),
      ('1184649345'),
      ('1669462420'),
      ('1003063280'),
      ('1114924834'),
      ('1013924372'),
      ('1225249865'),
      ('1043447253'),
      ('1003961251'),
      ('1073053757'),
      ('1366556227'),
      ('1285832634'),
      ('1609824010'),
      ('1023188851'),
      ('1083789630'),
      ('1336495910'),
      ('1891765178'),
      ('1033439732'),
      ('1700128592'),
      ('1194787218'),
      ('1235582925'),
      ('1477643690'),
      ('1104819366'),
      ('1477549756'),
      ('1205935012'),
      ('1235214834'),
      ('1568596765'),
      ('1669429577'),
      ('1275564098'),
      ('1649261462'),
      ('1235339227'),
      ('1750458485'),
      ('1275694184'),
      ('1154302727'),
      ('1003102781'),
      ('1083949382'),
      ('1164686879'),
      ('1013062769'),
      ('1336245828'),
      ('1235148594'),
      ('1689747552'),
      ('1760480503'),
      ('1366515488'),
      ('1083630073'),
      ('1679973364')
    as t(hco_npi)
  )
),

hco_engaged AS (
  SELECT DISTINCT
    b.npi__v AS npi
  FROM com_edp_prd.com_raw.vcrm_call2__v AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (
    SELECT DISTINCT TRY_CAST(hco_npi AS STRING)
    FROM base_table
  )
  and a.call_date__v between '2025-07-01' and '2025-12-31'
),

hco_profiled AS (
  SELECT DISTINCT
    b.npi__v AS npi
  FROM com_intgr.survey_target AS a
  LEFT JOIN com_intgr.customer AS b
    ON a.account__v = b.id
  WHERE TRY_CAST(b.npi__v AS STRING) IN (
    SELECT DISTINCT TRY_CAST(hco_npi AS STRING)
    FROM base_table
  )
)

select
  a.hco_npi,
  case when d.npi is not null then 1 else 0 end as is_hco_engaged,
  case when e.npi is not null then 1 else 0 end as is_hco_profiled
from base_table as a
left join hco_engaged as d
  on try_cast(a.hco_npi as string) = try_cast(d.npi as string)
left join hco_profiled as e
  on try_cast(a.hco_npi as string) = try_cast(e.npi as string)
;


In [0]:
select * from 78HCO_engaged_profiled

In [0]:
create or replace temp view 78HCO_engaged_profiled_final as
with base_table as (
  select *
  from (
    select * from values
      ('1467525790'),
      ('1023105400'),
      ('1649347469'),
      ('1144266024'),
      ('1003878539'),
      ('1760476659'),
      ('1578693321'),
      ('1396882205'),
      ('1639370059'),
      ('1326092404'),
      ('1295789907'),
      ('1013143213'),
      ('1912939703'),
      ('1346297843'),
      ('1235234535'),
      ('1053632463'),
      ('1376544320'),
      ('1144548322'),
      ('1750482022'),
      ('1598784555'),
      ('1659877280'),
      ('1265694442'),
      ('1013924182'),
      ('1093808040'),
      ('1285174649'),
      ('1114969169'),
      ('1932280666'),
      ('1063702785'),
      ('1164426896'),
      ('1669683512'),
      ('1093894131'),
      ('1215921457'),
      ('1548212988'),
      ('1437365186'),
      ('1184649345'),
      ('1669462420'),
      ('1003063280'),
      ('1114924834'),
      ('1013924372'),
      ('1225249865'),
      ('1043447253'),
      ('1003961251'),
      ('1073053757'),
      ('1366556227'),
      ('1285832634'),
      ('1609824010'),
      ('1023188851'),
      ('1083789630'),
      ('1336495910'),
      ('1891765178'),
      ('1033439732'),
      ('1700128592'),
      ('1194787218'),
      ('1235582925'),
      ('1477643690'),
      ('1104819366'),
      ('1477549756'),
      ('1205935012'),
      ('1235214834'),
      ('1568596765'),
      ('1669429577'),
      ('1275564098'),
      ('1649261462'),
      ('1235339227'),
      ('1750458485'),
      ('1275694184'),
      ('1154302727'),
      ('1003102781'),
      ('1083949382'),
      ('1164686879'),
      ('1013062769'),
      ('1336245828'),
      ('1235148594'),
      ('1689747552'),
      ('1760480503'),
      ('1366515488'),
      ('1083630073'),
      ('1679973364')
    as t(hco_npi)
  )
),

-- Customer name (one row per NPI if present in customer)
customer_name as (
  select
    b.npi__v as npi,
    b.hco_name_cda__v as customer_hco_name
  from com_intgr.customer b
  where try_cast(b.npi__v as string) in (
    select distinct try_cast(hco_npi as string)
    from base_table
  )

),

hco_engaged as (
  select
    b.npi__v as npi,
    a.entity_display_name__v as vcrm_hco_name
  from com_edp_prd.com_raw.vcrm_call2__v a
  left join com_intgr.customer b
    on a.account__v = b.id
  where try_cast(b.npi__v as string) in (
    select distinct try_cast(hco_npi as string)
    from base_table
  )
  -- and a.modified_date__v between '2025-07-01' and '2025-12-31'
  
),

hco_profiled as (
  select
    b.npi__v as npi,
    a.account_display_name__v as survey_hco_name
  from com_intgr.survey_target a
  left join com_intgr.customer b
    on a.account__v = b.id
  where try_cast(b.npi__v as string) in (
    select distinct try_cast(hco_npi as string)
    from base_table
  )

)

select distinct
  a.hco_npi,
  cn.customer_hco_name,
  e.vcrm_hco_name,
  p.survey_hco_name,
  case when e.npi is not null then 1 else 0 end as is_hco_engaged,
  case when p.npi is not null then 1 else 0 end as is_hco_profiled
from base_table a
left join customer_name cn
  on try_cast(a.hco_npi as string) = try_cast(cn.npi as string)
left join hco_engaged e
  on try_cast(a.hco_npi as string) = try_cast(e.npi as string)
left join hco_profiled p
  on try_cast(a.hco_npi as string) = try_cast(p.npi as string)
;


In [0]:
select * from 78HCO_engaged_profiled_final

In [0]:
create or replace temp view 78HCO_npi_names_simple as
with base_table as (
  select *
  from (
    select * from values
      ('1467525790'),
      ('1023105400'),
      ('1649347469'),
      ('1144266024'),
      ('1003878539'),
      ('1760476659'),
      ('1578693321'),
      ('1396882205'),
      ('1639370059'),
      ('1326092404'),
      ('1295789907'),
      ('1013143213'),
      ('1912939703'),
      ('1346297843'),
      ('1235234535'),
      ('1053632463'),
      ('1376544320'),
      ('1144548322'),
      ('1750482022'),
      ('1598784555'),
      ('1659877280'),
      ('1265694442'),
      ('1013924182'),
      ('1093808040'),
      ('1285174649'),
      ('1114969169'),
      ('1932280666'),
      ('1063702785'),
      ('1164426896'),
      ('1669683512'),
      ('1093894131'),
      ('1215921457'),
      ('1548212988'),
      ('1437365186'),
      ('1184649345'),
      ('1669462420'),
      ('1003063280'),
      ('1114924834'),
      ('1013924372'),
      ('1225249865'),
      ('1043447253'),
      ('1003961251'),
      ('1073053757'),
      ('1366556227'),
      ('1285832634'),
      ('1609824010'),
      ('1023188851'),
      ('1083789630'),
      ('1336495910'),
      ('1891765178'),
      ('1033439732'),
      ('1700128592'),
      ('1194787218'),
      ('1235582925'),
      ('1477643690'),
      ('1104819366'),
      ('1477549756'),
      ('1205935012'),
      ('1235214834'),
      ('1568596765'),
      ('1669429577'),
      ('1275564098'),
      ('1649261462'),
      ('1235339227'),
      ('1750458485'),
      ('1275694184'),
      ('1154302727'),
      ('1003102781'),
      ('1083949382'),
      ('1164686879'),
      ('1013062769'),
      ('1336245828'),
      ('1235148594'),
      ('1689747552'),
      ('1760480503'),
      ('1366515488'),
      ('1083630073'),
      ('1679973364')
    as t(hco_npi)
  )
)

select distinct
  bt.hco_npi as npi,
  c.hco_name_cda__v           as customer_hco_name,
  vc.entity_display_name__v   as vcrm_hco_name,
  st.account_display_name__v  as survey_hco_name
from base_table bt
left join com_intgr.customer c
  on try_cast(bt.hco_npi as string) = try_cast(c.npi__v as string)
left join com_edp_prd.com_raw.vcrm_call2__v vc
  on vc.account__v = c.id
left join com_intgr.survey_target st
  on st.account__v = c.id
;


In [0]:
select * from 78HCO_npi_names_simple

# HCO NAME COMPARISON

In [0]:
with hco_profiled as (
  select distinct
    b.npi__v as npi,
    b.hco_name_cda__v as customer_hco_name,
    a.account_display_name__v as survey_hco_name
  from com_intgr.survey_target a
  left join com_intgr.customer b
    on a.account__v = b.id
  where a.name__v = 'Denali HCO Survey'
)
select *
from hco_profiled;


In [0]:
with hco_profiled as (
  select distinct
    a.account__v as survey_account_id,
    b.npi__v as npi,
    b.hco_name_cda__v as crm_customer_hco_name,
    a.account_display_name__v as survey_hco_name,
    vc.entity_display_name__v as calls_table_hco_name
  from com_intgr.survey_target a
  left join com_intgr.customer b
    on a.account__v = b.id
  left join com_edp_prd.com_raw.vcrm_call2__v vc
    on vc.account__v = b.id
  where a.name__v = 'Denali HCO Survey'
)
select *
from hco_profiled;
